<a href="https://colab.research.google.com/github/estebancortes593/PROYECTO/blob/main/proyecto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')
ruta_archivo = '/content/drive/MyDrive/Programación/ride_stats.csv'
df = pd.read_csv(ruta_archivo)

df_limpio = df[df['is_archived'] == 0].copy()
df_limpio['year'] = pd.to_numeric(df_limpio['year'], errors='coerce')


def obtener_atracciones_parque(dataframe, parque, anio):
    filtro = (dataframe['park_name'] == parque) & (dataframe['year'] == anio)
    sub_df = dataframe[filtro]

    if sub_df.empty:
        return pd.DataFrame()

    tabla = sub_df.pivot_table(
        index='ride_name',
        columns='metric',
        values='minutes',
        aggfunc='mean'
    ).reset_index()

    return tabla.sort_values(by='ride_avg_wait', ascending=False).reset_index(drop=True)

def simular_itinerario(lista_seleccionadas, minutos_disponibles, tiempo_recorrido=15):

    atendidas_normal = []
    pendientes_normal = []
    atendidas_max = []
    pendientes_max = []

    tiempo_acum_normal = 0
    tiempo_acum_max = 0

    for atraccion in lista_seleccionadas:
        nombre = atraccion['nombre']
        # Sumamos los minutos de fila + minutos de duración/caminata entre atracciones
        costo_normal = atraccion['espera_promedio'] + tiempo_recorrido
        costo_max = atraccion['espera_maxima'] + tiempo_recorrido

        # Evaluación en Escenario Normal
        if tiempo_acum_normal + costo_normal <= minutos_disponibles:
            tiempo_acum_normal += costo_normal
            atendidas_normal.append((nombre, atraccion['espera_promedio']))
        else:
            pendientes_normal.append((nombre, atraccion['espera_promedio']))

        # Evaluación en Escenario de Alta Demanda (Máximo)
        if tiempo_acum_max + costo_max <= minutos_disponibles:
            tiempo_acum_max += costo_max
            atendidas_max.append((nombre, atraccion['espera_maxima']))
        else:
            pendientes_max.append((nombre, atraccion['espera_maxima']))

    return {
        'atendidas_normal': atendidas_normal,
        'pendientes_normal': pendientes_normal,
        'tiempo_normal': tiempo_acum_normal,
        'atendidas_max': atendidas_max,
        'pendientes_max': pendientes_max,
        'tiempo_max': tiempo_acum_max
    }

def graficar_comparacion(lista_seleccionadas):
    nombres = [item['nombre'][:22] + '...' if len(item['nombre']) > 22 else item['nombre'] for item in lista_seleccionadas]
    promedios = [item['espera_promedio'] for item in lista_seleccionadas]
    maximos = [item['espera_maxima'] for item in lista_seleccionadas]

    x = np.arange(len(nombres))
    ancho = 0.35

    fig, ax = plt.subplots(figsize=(9, 5))
    b1 = ax.bar(x - ancho/2, promedios, ancho, label='Escenario Normal (Promedio)', color='#2b5c8f')
    b2 = ax.bar(x + ancho/2, maximos, ancho, label='Escenario Alta Demanda (Máximo)', color='#d95f02')

    ax.set_ylabel('Minutos de Fila')
    ax.set_title('Comparación de Escenarios de Espera para tu Itinerario')
    ax.set_xticks(x)
    ax.set_xticklabels(nombres, rotation=25, ha='right')
    ax.legend()
    plt.tight_layout()
    plt.show()


print("BIENVENIDO AL PLANIFICADOR DE ITINERARIOS DISNEY")

# 1. Validación del año con ciclo WHILE
anoi = None
while anoi is None:
    ano = input("Elige el año de referencia que deseas revisar (2015 - 2026): ").strip()
    if ano.isdigit() and 2015 <= int(ano) <= 2026:
        anoi = int(ano)
    else:
        print("Respuesta inválida. Por favor ingresa un año numérico entre 2015 y 2026.")

# 2. Selección del parque usando una LISTA
parques_disponibles = list(df_limpio['park_name'].unique())
print("\nParques disponibles:")
for i, p in enumerate(parques_disponibles, 1):
    print(f"  {i}. {p}")

parque_elegido = None
while parque_elegido is None:
    op_parque = input("Ingresa el número del parque que visitarás: ").strip()
    if op_parque.isdigit() and 1 <= int(op_parque) <= len(parques_disponibles):
        parque_elegido = parques_disponibles[int(op_parque) - 1]
    else:
        print("Opción inválida. Intenta de nuevo.")

catalogo = obtener_atracciones_parque(df_limpio, parque_elegido, anoi)

if catalogo.empty:
    print(f"No se encontraron datos para {parque_elegido} en el año {anoi}.")
else:
    print(f"\n--- Atracciones activas en {parque_elegido} ({anoi}) ---")
    for idx, fila in catalogo.iterrows():
        print(f"  [{idx + 1}] {fila['ride_name']} | Promedio: {int(fila['ride_avg_wait'])} min | Máx: {int(fila['ride_max_wait'])} min")

    itinerario_usuario = []
    while True:
        seleccion = input("\nIngresa el número de la atracción para añadir a tu ruta (o escribe 'fin' para calcular): ").strip().lower()
        if seleccion == 'fin':
            if len(itinerario_usuario) > 0:
                break
            else:
                print("Debes elegir al menos una atracción antes de finalizar.")
                continue

        if seleccion.isdigit() and 1 <= int(seleccion) <= len(catalogo):
            fila_sel = catalogo.iloc[int(seleccion) - 1]
            itinerario_usuario.append({
                'nombre': fila_sel['ride_name'],
                'espera_promedio': fila_sel['ride_avg_wait'],
                'espera_maxima': fila_sel['ride_max_wait']
            })
            print(f"  -> Añadida: {fila_sel['ride_name']} (Total en lista: {len(itinerario_usuario)})")
        else:
            print("Número inválido. Elige un número de la lista o escribe 'fin'.")

    horas = float(input("\n¿Cuántas horas disponibles tienes en el parque para subir a atracciones? (ej. 4): "))
    minutos_disp = horas * 60

    resultado = simular_itinerario(itinerario_usuario, minutos_disp)

    print("\n" + "="*60)
    print(f"REPORTE DE VIABILIDAD DE ITINERARIO ({minutos_disp:.0f} MINUTOS DISPONIBLES)")
    print("="*60)

    print(f"\n1. ESCENARIO NORMAL (Tiempo requerido: {resultado['tiempo_normal']:.0f} min):")
    print(f"   - Atracciones que SÍ alcanzas a visitar ({len(resultado['atendidas_normal'])}): {[x[0] for x in resultado['atendidas_normal']]}")
    print(f"   - Atracciones que quedan PENDIENTES ({len(resultado['pendientes_normal'])}): {[x[0] for x in resultado['pendientes_normal']]}")

    print(f"\n2. ESCENARIO ALTA DEMANDA (Tiempo requerido: {resultado['tiempo_max']:.0f} min):")
    print(f"   - Atracciones que SÍ alcanzas a visitar ({len(resultado['atendidas_max'])}): {[x[0] for x in resultado['atendidas_max']]}")
    print(f"   - Atracciones que quedan PENDIENTES ({len(resultado['pendientes_max'])}): {[x[0] for x in resultado['pendientes_max']]}")

    fila_total_max = sum([x['espera_maxima'] for x in itinerario_usuario])
    ahorro_minutos = fila_total_max * 0.75
    print(f"\n3. RECOMENDACIÓN DE NEGOCIO (PASE RÁPIDO / LIGHTNING LANE):")
    print(f"   - En un día de alta demanda, comprar el Pase Rápido te ahorraría ~{ahorro_minutos:.0f} minutos de fila ({ahorro_minutos/60:.1f} horas).")
    if len(resultado['pendientes_max']) > 0:
        print("   - Veredicto: ¡SÍ te conviene adquirir el Pase Rápido! Sin él no lograrás completar tu lista de atracciones.")
    else:
        print("   - Veredicto: NO necesitas pagar el Pase Rápido; tu tiempo es suficiente incluso en el escenario de alta demanda.")

    graficar_comparacion(itinerario_usuario)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BIENVENIDO AL PLANIFICADOR DE ITINERARIOS DISNEY


KeyboardInterrupt: Interrupted by user